In [1]:
# get a list of file names from the gis folder
import os
file_names = os.listdir("data/pdd")
# only get the names ending with pdf
file_names = [f for f in file_names if f.endswith(".pdf")]
# remove the file name extension
file_names = [f.replace(".pdf", "") for f in file_names]
file_names = [f.replace("_", "") for f in file_names]
print(file_names)

['VCS3226', 'VCS5283', 'VCS4782', 'VCS5458']


In [2]:
import pandas as pd
berkley = pd.read_excel("data/berkleydata/Voluntary-Registry-Offsets-Database--v2024-12-year-end.xlsx", sheet_name="PROJECTS")
# load the second tab

In [19]:
berkley_sm = berkley[[ 'Project ID', 'Country', 'Scope',
    'Voluntary Status',' Type','Reduction / Removal','Methodology / Protocol', 'Project Developer',
    "Total Credits \nIssued", 'Total Credits Remaining', 'Total Buffer \nPool Deposits'
    ]]

In [21]:
berkley_sm = berkley_sm.query("Country == 'Indonesia'").query("Scope == 'Forestry & Land Use'")
# berkley_sm['Project ID'] = berkley_sm['Project ID'].apply(lambda x: x.replace('VCS', ''))
berkley_sm.head(15)

,Project ID,Country,Scope,Voluntary Status,Type,Reduction / Removal,Methodology / Protocol,Project Developer,Total Credits \nIssued,Total Credits Remaining,Total Buffer \nPool Deposits
6101,VCS674,Indonesia,Forestry & Land Use,Registered,REDD+,Reduction,VM0004 Methodology for Conservation Projects t...,InfiniteEARTH,33625616.0,7583981.0,4376475.0
6894,VCS1477,Indonesia,Forestry & Land Use,Registered,REDD+,Reduction,VM0007 REDD+ Methodology Framework (REDD-MF),PT. Rimba Makmur Utama (PT. RMU),39998290.0,10202169.0,4216585.0
6908,VCS1493,Indonesia,Forestry & Land Use,Registered,Wetland Restoration,Mixed,AR-AM0014 Afforestation and reforestation of d...,Livelihoods Fund SICAV SIF,397071.0,137379.0,48955.0
6913,VCS1498,Indonesia,Forestry & Land Use,Under validation,REDD+,Reduction,VM0007 REDD+ Methodology Framework (REDD-MF),"KPHP Tasik Besar Serkap (KPHP TBS), Riau Provi...",0.0,0.0,0.0
7204,VCS1899,Indonesia,Forestry & Land Use,Registered,REDD+,Reduction,VM0007 REDD+ Methodology Framework (REDD-MF),Multiple Proponents,2638073.0,130997.0,439310.0
7519,VCS2395,Indonesia,Forestry & Land Use,Registration requested,REDD+,Reduction,AM0014 Natural gas-based package cogeneration;...,Multiple Proponents,0.0,0.0,0.0
7527,VCS2403,Indonesia,Forestry & Land Use,Registered,REDD+,Reduction,VM0007 REDD+ Methodology Framework (REDD-MF),Multiple Proponents,0.0,0.0,0.0
8051,VCS3012,Indonesia,Forestry & Land Use,Registration requested,Afforestation/Reforestation,Impermanent Removal,AR-AMS0007 Afforestation and reforestation pro...,The PURE PROJECT SAS,0.0,0.0,0.0
8244,VCS3226,Indonesia,Forestry & Land Use,Under validation,REDD+,Reduction,VM0007 REDD+ Methodology Framework (REDD-MF),Multiple Proponents,0.0,0.0,0.0
8554,VCS3587,Indonesia,Forestry & Land Use,Inactive,Afforestation/Reforestation,Impermanent Removal,AR-AMS0007 Afforestation and reforestation pro...,The PURE PROJECT SAS,0.0,0.0,0.0


In [5]:
berkley_sm = berkley_sm[berkley_sm['Project ID'].isin(file_names)]

In [6]:
berkley_sm = berkley_sm.rename(
    columns = {
        'Project ID': 'project_code', #exist
        'Voluntary Status': 'status', #exist
        ' Type': 'type',
        'Reduction / Removal': 'reduction_removal',
        'Methodology / Protocol': 'methodology', #exist
        'Project Developer': 'project_developer',
        "Total Credits \nIssued": 'total_credits', #exist
        'Total Credits Remaining': 'remaining_credits', #exist
        'Total Buffer \nPool Deposits': 'buffer'
    }
)

In [7]:
berkley_sm['project_code'] = berkley_sm['project_code'].apply(lambda x: x.replace('VCS', ''))
berkley_sm

,project_code,Country,Scope,status,type,reduction_removal,methodology,project_developer,total_credits,remaining_credits,buffer
8244,3226,Indonesia,Forestry & Land Use,Under validation,REDD+,Reduction,VM0007 REDD+ Methodology Framework (REDD-MF),Multiple Proponents,0.0,0.0,0.0
9528,4782,Indonesia,Forestry & Land Use,Under validation,REDD+,Reduction,VM0007 REDD+ Methodology Framework (REDD-MF),PT Nusantara Raya Solusi,0.0,0.0,0.0
9870,5283,Indonesia,Forestry & Land Use,Under validation,Improved Forest Management,Mixed,VM0010 Methodology for Improved Forest Managem...,PT Strata Pacific,0.0,0.0,0.0


columns to add: "Project Developer", "Registry Documents", "Type", "Reduction / Removal", "Methodology / Protocol", 

In [8]:
os.chdir('/Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python')
from database import supabase

In [ ]:
df = berkley_sm.copy()
df.head()

In [17]:
success_count = 0
error_count = 0
for _, row in df.iterrows():
    project_code = row["project_code"]
    
    # Convert row to dict and remove project_code as it's the matching field
    update_data = row.to_dict()
    del update_data["project_code"]
    del update_data["Country"]
    del update_data["Scope"]
    
    try:
        # Update the project in Supabase where project_code matches
        result = supabase.table("projects").update(update_data).eq("project_code", project_code).execute()
        
        # Check if the update was successful
        if result.data:
            print(f"Successfully updated project {project_code}")
            success_count += 1
        else:
            print(f"No project found with code {project_code}")
            error_count += 1
            
    except Exception as e:
        print(f"Error updating project {project_code}: {str(e)}")
        error_count += 1

Successfully updated project 3226
Successfully updated project 4782
Successfully updated project 5283


In [ ]:
Scope
type
reduction_removal
methodology
project_developer
total_credits
remaining_credits
buffer